In [8]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
import re
from scipy.stats import spearmanr, kendalltau

import sys
sys.path.insert(1, "../")

from dependencies.data_generator import load_dataset, load_dataset_on_inference
from dependencies.train_config import TrainConfig

from train_surrogate import SurrogateTrainer
from inference_surrogate import InferSurrogate

from tqdm.auto import tqdm
np.random.seed(42)

In [9]:
params = json.loads(
    Path(
        "/home/alexander/RAS/m1p/predicator-function-for-neural-networks/code/configs/surrogate_hp_CIFAR100.json"
    ).read_text()
)
config = TrainConfig(**params)
load_dataset(config)

Loading 2925 random DIV models: 100%|██████████| 2925/2925 [00:00<00:00, 1616594.97it/s]


In [10]:
config.models_dict_path = []
config.dataset_path = Path(config.dataset_path)
for file_path in config.dataset_path.rglob("*.json"):
    config.models_dict_path.append(file_path)

In [11]:
initial_archs = []
for arch_json_path in tqdm(config.models_dict_path, desc="Loading pretrained archs"):
    arch = json.loads(arch_json_path.read_text(encoding="utf-8"))
    arch["id"] = int(re.search(r"model_(\d+)", str(arch_json_path)).group(1))
    initial_archs.append(arch)

Loading pretrained archs:   0%|          | 0/2925 [00:00<?, ?it/s]

In [12]:
accs = [arch["valid_accuracy"] for arch in initial_archs]

mean_acc = np.mean(accs
)
var_acc = np.var(accs)
print("Mean accuracy:", mean_acc)
print("sqrt of variance of accuracy:", np.sqrt(var_acc))

Mean accuracy: 0.37633323076923075
sqrt of variance of accuracy: 0.011134773570581177


In [13]:
trainer = SurrogateTrainer(config)
trainer.get_diversity_matrix()

100%|██████████| 2925/2925 [00:01<00:00, 2902.23it/s]


In [14]:
matrix = np.array(trainer.config.diversity_matrix)
n = matrix.shape[0]

# Извлекаем только внедиагональные элементы
# mask создает булеву матрицу, где True только там, где индексы i != j
mask = ~np.eye(n, dtype=bool)
off_diagonal_elements = matrix[mask]

# Считаем метрики
mean_diversity = np.mean(off_diagonal_elements)
variance_diversity = np.var(off_diagonal_elements)

print(f"Среднее разнообразие (overlap): {mean_diversity}")
print(f"Среднее отклонение: {np.sqrt(variance_diversity)}")

Среднее разнообразие (overlap): 0.32445938672582936
Среднее отклонение: 0.007867905663530222
